# **Yellow Taxi Trip Analysis**

**Author:** Ignacio Bavastro | **Data sources:** TLC of NYC Yellow Taxi Trip Records (2018–2023) and Taxi Zone Lookup | **Date:** June 2026

## **Executive Summary**

This assignment analyzes TLC NYC Yellow Taxi trip records from 2018 through January 2023, providing a multi-year view of trip demand, revenue patterns, and operational behavior across the city. The analysis focuses on identifying key commercial trends, segment differences, and data quality considerations that are relevant for business reporting and decision-making.

From a business perspective, the main takeaway is that the taxi market is not homogeneous. It is composed of distinct commercial segments with different behaviors, economics, and operational needs. Performance should therefore be interpreted through trip patterns, since location and timing have a direct impact on both volume and revenue. This is reflected in four clear segments: airport trips, Manhattan-Manhattan, Non-Manhattan-Manhattan, and Non-Manhattan-Non-Manhattan.

From a Data Quality perspective, there are also important issues to keep in mind. congestion_surcharge field appears to require further validation, and there may be additional concerns related to sensor reliability and the store-and-forward process. These elements should be reviewed carefully, as they may affect the integrity of KPI reporting and the robustness of any conclusions drawn from the data.

The table below summarizes the most relevant findings and the associated business actions

### Key Findings

| # | Insight | Recommended Actions |
|---|---|---|
| 1 | **NYC taxi demand is concentrated in short Manhattan rides** with clear weekday commuting peaks. | The core business is highly urban and time-sensitive, so driver allocation should prioritize Manhattan and peak weekday hours. |
| 2 | **Airport trips are a distinct high-value segment** with much longer duration, distance, and fare levels. | Airport trips should be reported separately and managed as a premium operational segment rather than mixed with general urban trips. |
| 3 | **Manhattan-to-Manhattan, Manhattan-to-outer borough, and non-Manhattan trips behave differently.** | Trip segmentation should be embedded in reporting to avoid masking distinct demand and pricing patterns. |
| 4 | **Trip count has declined over time, while total amount has increased.** | Revenue growth is being driven more by trip distance and pricing than by stronger demand, so trip volume is the better KPI for market health. |
| 5 | **Vendor concentration is very high, with one dominant player and one main competitor.** | The market shows concentration risk, and competitive analysis should monitor whether dominance continues to increase. |
| 6 | **Credit card trips show strong recorded tipping behaviour, while cash tips are not captured.** | Tip KPIs are incomplete by design, so cash trips should be excluded from tip analysis and results should be interpreted as partial rather than total market behaviour. |
| 7 | **`congestion_surcharge` appears inconsistently recorded, and the store-and-forward pattern suggests possible system issues.** | Data quality rules should be strengthened, and the surcharge and capture process should be reviewed before relying on these fields for revenue analysis. |
| 8 | **Some extreme records may reflect sensor or capture problems rather than real trips.** | Outlier handling should remain conservative, and suspicious records should be monitored separately instead of being treated as normal business activity. |
| 9 | **The pandemic created a clear break in demand in 2020.** | Historical trend analysis should treat 2020 as an exceptional period, not a normal reference year. |

## 0. Libraries Imports

In [1]:
import pandas as pd
from pandasql import sqldf
import numpy as np

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Data Load and schema validation

Revelevant data sources to load are:
- **Yellow Taxi trips**: fact table with a list of all taxi trips to analise
- **NYC TLC taxi zone lookup**: dim table that contains a list of all zones where rides can start/finish and its fields

### Yellow Taxi trips

In [2]:
# Load Yellow Taxi trips table and convert dates
fact_trips = pd.read_csv(r'Datasources\Input\Yellow_Taxi_Assignment.csv',parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])


print(f"Yellow Taxi Trips table loaded successfully")
print(f"Yellow Taxi Trips table shape: {fact_trips.shape[0]:,} rows and {fact_trips.shape[1]} columns")
print(f"Yellow Taxi Trips table date range: {fact_trips['tpep_pickup_datetime'].dt.date.min()} to {fact_trips['tpep_pickup_datetime'].dt.date.max()}")

Yellow Taxi Trips table loaded successfully
Yellow Taxi Trips table shape: 304,978 rows and 19 columns
Yellow Taxi Trips table date range: 2018-01-01 to 2023-01-31


In [3]:
# Show dataframe schema and check which columns need to be converted

fact_trips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304978 entries, 0 to 304977
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               304978 non-null  int64         
 1   tpep_pickup_datetime   304978 non-null  datetime64[ns]
 2   tpep_dropoff_datetime  304978 non-null  datetime64[ns]
 3   passenger_count        295465 non-null  float64       
 4   trip_distance          304978 non-null  float64       
 5   RatecodeID             295465 non-null  float64       
 6   store_and_fwd_flag     295465 non-null  object        
 7   PULocationID           304978 non-null  int64         
 8   DOLocationID           304978 non-null  int64         
 9   payment_type           304978 non-null  int64         
 10  fare_amount            304978 non-null  float64       
 11  extra                  304978 non-null  float64       
 12  mta_tax                304978 non-null  floa

In [4]:
# Convert numeric columns and coerce values

numeric_cols = [
    'fare_amount', 'tip_amount', 'tolls_amount', 'total_amount',
    'trip_distance', 'extra', 'mta_tax', 'improvement_surcharge',
    'congestion_surcharge', 'airport_fee'
]

for columns in numeric_cols:
    fact_trips[columns] = pd.to_numeric(fact_trips[columns], errors='coerce')

In [5]:
# Show extract of records
fact_trips.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2018-01-01 12:02:01,2018-01-01 12:04:05,1.0,0.53,1.0,N,142,163,1,3.5,0.0,0.5,1.29,0.0,0.3,5.59,NaN,NaN
1,2,2018-01-01 12:26:48,2018-01-01 12:31:29,1.0,1.05,1.0,N,140,236,1,6.0,0.0,0.5,1.02,0.0,0.3,7.82,NaN,NaN
2,2,2018-01-01 01:28:34,2018-01-01 01:39:38,4.0,1.83,1.0,N,211,158,1,9.5,0.5,0.5,1.62,0.0,0.3,12.42,NaN,NaN
3,1,2018-01-01 08:51:59,2018-01-01 09:01:45,1.0,2.30,1.0,N,249,4,2,10.0,0.0,0.5,0.00,0.0,0.3,10.80,NaN,NaN
4,2,2018-01-01 01:00:19,2018-01-01 01:14:16,1.0,3.06,1.0,N,186,142,1,12.5,0.5,0.5,1.00,0.0,0.3,14.80,NaN,NaN


In [6]:
# Get an overview of all columns. 
# With this information it is already visible that there are Data Quality issues, as there trips that have negative values for trip_distance, fare_amount, extra, mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee fields.
fact_trips.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,304978.0,NaN,NaN,NaN,1.672786,1.0,1.0,2.0,2.0,6.0,0.514069
tpep_pickup_datetime,304978,NaN,NaN,NaN,2020-07-16 11:54:50.074785792,2018-01-01 00:25:49,2019-04-08 09:18:22.500000,2020-07-17 17:01:39,2021-10-24 11:29:50.500000,2023-01-31 23:57:28,NaN
tpep_dropoff_datetime,304978,NaN,NaN,NaN,2020-07-16 12:11:31.905307648,2018-01-01 00:38:59,2019-04-08 09:32:34.750000128,2020-07-17 17:17:55,2021-10-24 11:42:53.249999872,2023-02-01 00:09:12,NaN
passenger_count,295465.0,NaN,NaN,NaN,1.479126,0.0,1.0,1.0,2.0,6.0,1.108255
trip_distance,304978.0,NaN,NaN,NaN,4.587589,-16.86,1.0,1.73,3.21,177247.4,434.226624
RatecodeID,295465.0,NaN,NaN,NaN,1.142931,1.0,1.0,1.0,1.0,99.0,2.969941
store_and_fwd_flag,295465,2,N,292425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,304978.0,NaN,NaN,NaN,163.744975,1.0,121.0,162.0,234.0,265.0,66.57049
DOLocationID,304978.0,NaN,NaN,NaN,160.988898,1.0,107.0,162.0,234.0,265.0,70.975905
payment_type,304978.0,NaN,NaN,NaN,1.240463,0.0,1.0,1.0,2.0,5.0,0.528257


### NYC TLC taxi zone lookup

In [7]:
# Load Zones table and convert dates
dim_zones = pd.read_csv(r'Datasources\Input\taxi_zone_lookup.csv')

print(f"Yellow Taxi Trips table loaded successfully")
print(f"Zone lookup shape: {dim_zones.shape[0]} rows and {dim_zones.shape[1]} columns")

Yellow Taxi Trips table loaded successfully
Zone lookup shape: 265 rows and 4 columns


In [8]:
# Show dataframe schema and check which columns need to be converted
dim_zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [9]:
# Show extract of records
dim_zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [10]:
# Get an overview of all columns

dim_zones.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
LocationID,265.0,NaN,NaN,NaN,133.0,76.643112,1.0,67.0,133.0,199.0,265.0
Borough,264,7,Queens,69,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zone,264,261,Governor's Island/Ellis Island/Liberty Island,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
service_zone,263,4,Boro Zone,205,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Data Quality Assesment and Outlier Detection


Before starting the analysis, the data is validated to assess whether it is analytically trustworthy.

The analysis is divided into two steps:
- Data Quality Assesment: identifies missing values and clearly invalid records that violate business rules, such as negative distances or impossible totals. (eg. negative distance values)
- Outlier Detection: identifies extreme but still technically valid observations that may distort averages, trend lines, or other summary metrics (eg. trip with a duration longer than 2 hours)

The dataset is broadly usable, but it contains a small set of quality issues that should be addressed before deeper analysis. The main concern appears to be that `congestion_surcharge` is not consistently recorded, which can affect the accuracy of fare reconciliation and revenue-related metrics. Given the volume of affected records (~20% of total records), it may be worth an operational review to understand whether drivers or the capture process need additional guidance or incentives to ensure the surcharge is properly reported.

In a smaller number of cases, the `store_and_fwd_flag` pattern suggests a possible issue in the store-and-forward process. In addition, there are records with clearly invalid values, which should be investigated to understand whether they result from data entry issues, system errors, or cancelled trips that were not properly classified.

Outlier detection also showed a small number of extreme trips with unusually long durations, excessive distances, or implausibly high speeds, which were excluded from the analytical dataset to prevent distortion of summary metrics and segment-level comparisons. They may reflect data capture issues such as problem with sensors, and it would be worth investigating their root cause.

### 2.A Data Quality Assesment

The data quality assessment shows that the dataset is broadly usable, but it still requires targeted cleaning before deeper analysis. Missing values are limited and mostly concentrated in a small subset of fields, while invalid records represent a small share of the data and will be excluded to preserve analytical integrity.

The main issue is not completeness, but consistency. A significant share of records shows mismatches between `total_amount` and its fare components, and the pattern is systematic rather than random, for this reason records that are not consistent weren't excluded from the analysis. This suggests that the issue is likely driven by missing surcharge information, especially `congestion_surcharge`, and should therefore be treated as a revenue reporting caveat rather than a true fare error. For this reason analysis including `congestion_surcharge` should be done more carefully.

#### 2.A.I Missing Values

The dataset shows missing values in only five columns, which suggests that completeness is generally strong. The missing values in `store_and_fwd_flag`, `RatecodeID`, and `passenger_count` occur in the same records, indicating a likely issue in "store and forward" system, rather than independent null patterns. These represent only 3% of total trips.

The remaining missing values appear in `airport_fee` and `congestion_surcharge`, which are not applicable to every trip, and that might be the reason why they are not informed in many cases. _This topic is later addressed in `Records with inconsistent fare components` subsection_

In [11]:
missing = fact_trips.isnull().sum()
missing_pct = (missing / len(fact_trips) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report[missing_report['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

,Missing Count,Missing %
airport_fee,198761,65.17
congestion_surcharge,72632,23.82
passenger_count,9513,3.12
RatecodeID,9513,3.12
store_and_fwd_flag,9513,3.12


#### 2.A.II Invalid Values

Roughly 2% of the records contain invalid values or appear to represent cancelled trips, as several key fields are recorded as zero. From an analytical perspective, these rows do not reflect meaningful trip activity and would introduce noise into the results if kept in the dataset.

For that reason, they will be excluded from the analysis. This helps ensure that the findings are based on valid, comparable trip records and that summary metrics are not distorted by transactions that were never properly completed.

In [12]:
fact_trips['trip_duration_min'] = (fact_trips['tpep_dropoff_datetime'] - fact_trips['tpep_pickup_datetime']).dt.total_seconds() / 60

raw_count = len(fact_trips)


cleaning_mask = (
    (fact_trips['trip_duration_min'] > 0) &
    (fact_trips['trip_distance'] > 0) &
    (fact_trips['fare_amount'] > 0) &
    ~(fact_trips['extra'] < 0) &
    ~(fact_trips['mta_tax'] < 0) &
    ~(fact_trips['tip_amount'] < 0) &
    ~(fact_trips['tolls_amount'] < 0) &
    ~(fact_trips['improvement_surcharge'] < 0) &
    ~(fact_trips['total_amount'] < 0) &
    ~(fact_trips['congestion_surcharge'] < 0) &
    ~(fact_trips['airport_fee'] < 0)
)


fact_trips_cleaned = fact_trips[cleaning_mask].copy()
removed = raw_count - len(fact_trips_cleaned)

print(f"Raw records   : {raw_count:,}")
print(f"Clean records : {len(fact_trips_cleaned):,}")
print(f"Removed       : {removed:,} ({removed/raw_count*100:.2f}% of total)")

Raw records   : 304,978
Clean records : 299,831
Removed       : 5,147 (1.69% of total)


In [13]:
fact_trips["trip_duration_min"].describe()

count    304978.000000
mean         16.697175
std          59.903429
min         -44.133333
25%           6.600000
50%          11.016667
75%          18.050000
max        1439.866667
Name: trip_duration_min, dtype: float64

Data Quality Business Rules

Based on the results of the data quality assessment, the most frequently violated business rule is Zero or Negative trip_distance, which suggests that a significant number of trips were recorded with no distance traveled or with invalid negative distance values.

- **Negative total_amount**: Flags trips where `total_amount < 0`, since the total amount charged for a trip should never be negative.
- **Zero or Negative trip_duration**: Flags trips where `trip_duration_min <= 0` and `total_amount > 0`, since a paid trip should always have a positive duration.
- **Zero or Negative trip_distance**: Flags trips where `trip_distance <= 0` and `total_amount > 0`, since a paid trip should always cover a positive distance.
- **Zero or Negative fare_amount**: Flags trips where `fare_amount <= 0` and `total_amount > 0`, since a trip with a positive total should also have a positive base fare.
- **Negative extra**: Flags trips where `extra < 0`, since extra charges and surcharges should not be negative.
- **Negative mta_tax**: Flags trips where `mta_tax < 0`, since MTA tax is a mandatory fee and cannot be negative.
- **Negative tip_amount**: Flags trips where `tip_amount < 0`, since tip amounts can be zero but should never be negative.
- **Negative improvement_surcharge**: Flags trips where `improvement_surcharge < 0`, since this surcharge is a fixed fee and cannot be negative.
- **Negative tolls_amount**: Flags trips where `tolls_amount < 0`, since toll charges should only increase the total trip cost.
- **Negative congestion_surcharge**: Flags trips where `congestion_surcharge < 0`, since congestion fees are additive charges and cannot be negative.
- **Negative airport_fee**: Flags trips where `airport_fee < 0`, since airport-related fees should never be recorded as negative values.


![alt text](Images\dq_issues_n_records.png)

#### 2.A.III Records with Inconsistent Fare Components

According to the data dictionary, `total_amount` should represent the full charge to the passenger and should be consistent with the sum of the fare, surcharges, tips, and tolls.

The analysis shows that ~25% of records present inconsistencies between `total_amount` and the sum of their components. In most cases, the difference is exactly USD 2.50, suggesting that the issue is systematic rather than random. This pattern strongly points to missing `congestion_surcharge` values, as USD 2.50 is the most common surcharge amount in the affected records.

From an analytical standpoint, this is important because it means the mismatch is likely driven by incomplete component reporting rather than incorrect total fares. As a result, these records should be interpreted carefully, and this issue should be considered when analyzing revenue. Because this issue is mainly only affecting `congestion_surcharge` field, these records are not excluded from the analysis.

In [14]:
# Sum the fare components, treating missing values as zero
fact_trips_cleaned['sum_components'] = (
    fact_trips_cleaned['fare_amount'].fillna(0) +
    fact_trips_cleaned['extra'].fillna(0) +
    fact_trips_cleaned['mta_tax'].fillna(0) +
    fact_trips_cleaned['tip_amount'].fillna(0) +
    fact_trips_cleaned['tolls_amount'].fillna(0) +
    fact_trips_cleaned['improvement_surcharge'].fillna(0) +
    fact_trips_cleaned['congestion_surcharge'].fillna(0) +
    fact_trips_cleaned['airport_fee'].fillna(0)
)

# Compare the summed components with the reported total amount
fact_trips_cleaned['amount_diff'] = fact_trips_cleaned['total_amount'] - fact_trips_cleaned['sum_components']
fact_trips_cleaned['amount_diff'] = fact_trips_cleaned['amount_diff'].round(2)

# Flag rows with differences above the allowed tolerance
tolerance = 0.01
fact_trips_cleaned['amount_mismatch_flag'] = fact_trips_cleaned['amount_diff'].abs() > tolerance

# Count inconsistent records
inconsistent_fare_components = len(fact_trips_cleaned[fact_trips_cleaned['amount_mismatch_flag'] == True])

print(f"Amount of records with inconsistent fare components: {inconsistent_fare_components:,} ({inconsistent_fare_components/raw_count*100:.2f}% of total)")

Amount of records with inconsistent fare components: 73,008 (23.94% of total)


In [15]:
print("Summary statistics for the amount differences in records where total_amount does not match the sum of the fare components.\n"
"This helps understand the size and spread of the mismatches\n")

print(fact_trips_cleaned[fact_trips_cleaned["amount_mismatch_flag"] == True][["amount_diff"]].describe().T)

print("\n\n\n")

print("Summary statistics for the congestion surcharge column.\n"
"This shows that the most common congestion surcharge value is USD 2.50, which aligns with most of the mismatches.\n")
print(fact_trips_cleaned[["congestion_surcharge"]].describe().T)



Summary statistics for the amount differences in records where total_amount does not match the sum of the fare components.
This helps understand the size and spread of the mismatches

               count      mean       std   min  25%  50%  75%   max
amount_diff  73008.0 -2.171894  1.255789 -3.75 -2.5 -2.5 -2.5  17.5




Summary statistics for the congestion surcharge column.
This shows that the most common congestion surcharge value is USD 2.50, which aligns with most of the mismatches.

                         count      mean      std  min  25%  50%  75%   max
congestion_surcharge  228083.0  2.282625  0.70441  0.0  2.5  2.5  2.5  2.75


##### More than 90% of the discrepancies are exactly USD 2.50, which is the most common value for `congestion_surcharge`. This strongly suggests that `congestion_surcharge` is not consistently being recorded, which is consistent with the missing-value pattern observed earlier.

In [16]:
fact_trips_cleaned[fact_trips_cleaned["amount_mismatch_flag"] == True][["amount_diff"]].value_counts()

amount_diff
-2.50          66377
 2.50           4309
-3.75            921
-1.25            880
 1.95            455
 5.00             25
 3.90             13
 4.95             13
-2.80              6
 4.45              4
-0.30              1
 0.75              1
 1.94              1
 9.90              1
 17.50             1
Name: count, dtype: int64

### 2.B Outlier detection

To preserve the analytical integrity of the dataset, a conservative outlier filter is applied based on trip duration, trip distance, and speed. The objective is not to remove rare but valid trips, but to reduce the influence of extreme values and potential data quality issues on the analysis.

Trips longer than 2 hours, longer than 40 miles, or with speeds above 70 mph were excluded, as these observations are highly unlikely to represent normal urban taxi operations and can disproportionately distort descriptive statistics and downstream insights.

In [17]:
fact_trips_cleaned['speed_mph'] = (fact_trips_cleaned['trip_distance'] /(fact_trips_cleaned['trip_duration_min'] / 60))    

outlier_metrics = fact_trips_cleaned[['trip_duration_min', 'trip_distance', 'speed_mph']].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

outlier_metrics.round(2)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
trip_duration_min,299831.0,16.80,60.19,0.02,1.67,3.15,6.70,11.08,18.08,36.95,60.98,1439.87
trip_distance,299831.0,4.65,437.94,0.01,0.30,0.50,1.04,1.76,3.26,11.83,19.55,177247.40
speed_mph,299831.0,19.64,1279.22,0.01,3.07,4.92,7.93,10.55,14.37,25.43,36.75,462384.52


#### Outlier filtering rationale

The thresholds were defined as conservative operational limits, taking into account that different trip segments may follow different behavioral patterns. The objective was to preserve rare but plausible trips while excluding only extreme observations likely to distort the analysis.

The thresholds were set at approximately twice the 99th percentile. The following rules were applied:

- Trips longer than 121.96 minutes represent extreme duration values for a NYC taxi ride.
- Trips longer than 39.1 miles are highly atypical for this business context.
- Implied speeds above 73.5 mph are inconsistent with normal urban taxi operations and usually indicate data quality issues.

Using these three dimensions together is more robust than relying on a single rule, because it validates both absolute values and consistency between distance and duration.

In [18]:
n_outliers = fact_trips_cleaned[(
    (fact_trips_cleaned['trip_duration_min'] >= 121.96) |
    (fact_trips_cleaned['trip_distance']     >= 39.1)  |
    (fact_trips_cleaned['speed_mph']         >= 73.5))]

print(f"There's a total of {len(n_outliers):,} ({len(n_outliers)/len(fact_trips_cleaned)*100:.2f}% of total records) outliers to be removed")

There's a total of 1,007 (0.34% of total records) outliers to be removed


In [19]:
fact_trips_cleaned = fact_trips_cleaned[~(
    (fact_trips_cleaned['trip_duration_min'] >= 121.96) |
    (fact_trips_cleaned['trip_distance']     >= 39.1)  |
    (fact_trips_cleaned['speed_mph']         >= 73.5))]

print("Outliers successfully removed")

Outliers successfully removed


## 3. Data preparation and enrichment

The trip table is enriched with derived fields and taxi zone details to support further analysis.

### 3.A Enrich Trips table with Taxi Zone Data

Merge the trip data with the official taxi zone lookup to add borough and zone-level context. 
Prior to merge the datasets it is vital to ensure that every pickup and dropoff location ID exists in the taxi zone lookup table (referential integrity), and to confirm that the taxi zone key is unique before merging the datasets.


#### Check Referential Integrity - Validate that every pickup and dropoff location ID exists in the taxi zone lookup table.

In [20]:
# Check referential integrity for pickup location IDs
df_referential_integrity_PU = fact_trips_cleaned.merge(dim_zones, how='left', left_on='PULocationID',right_on="LocationID")
missing_pu = df_referential_integrity_PU[df_referential_integrity_PU["LocationID"].isnull()]

print("Pickup location referential integrity check")
print(f"Rows with missing pickup zone references: {len(missing_pu)}")


# Check referential integrity for dropoff locations
df_referential_integrity_DO = fact_trips_cleaned.merge(dim_zones, how='left', left_on='DOLocationID',right_on="LocationID")
missing_do = df_referential_integrity_DO[df_referential_integrity_DO["LocationID"].isnull()]

print("\nDropoff location referential integrity check")
print(f"Rows with missing dropoff zone references: {len(missing_do)}")

Pickup location referential integrity check
Rows with missing pickup zone references: 0

Dropoff location referential integrity check
Rows with missing dropoff zone references: 0


#### Verify that LocationID is the primary key and contains unique values, so it does not introduce duplicate trips into the fact table.

In [21]:
dim_zones["LocationID"].value_counts()

LocationID
1      1
183    1
169    1
170    1
171    1
      ..
95     1
96     1
97     1
98     1
265    1
Name: count, Length: 265, dtype: int64

#### Merge datasets

In [22]:
pickup_zones = dim_zones.rename(columns={
    'LocationID': 'PULocationID',
    'Borough': 'PU_Borough',
    'Zone': 'PU_Zone',
    'service_zone': 'PU_service_zone'
})

dropoff_zones = dim_zones.rename(columns={
    'LocationID': 'DOLocationID',
    'Borough': 'DO_Borough',
    'Zone': 'DO_Zone',
    'service_zone': 'DO_service_zone'
})

fact_trips_cleaned = fact_trips_cleaned.merge(pickup_zones, how='left', on='PULocationID')
fact_trips_cleaned = fact_trips_cleaned.merge(dropoff_zones, how='left', on='DOLocationID')

print('Zone lookup merged successfully.')

Zone lookup merged successfully.


### 3.B Add extra columns

Enrich the dataset with additional derived fields to support deeper analysis.

In [23]:
# Time-based fields
fact_trips_cleaned['pickup_hour'] = fact_trips_cleaned['tpep_pickup_datetime'].dt.hour
fact_trips_cleaned['pickup_dow'] = fact_trips_cleaned['tpep_pickup_datetime'].dt.dayofweek
fact_trips_cleaned['pickup_month'] = fact_trips_cleaned['tpep_pickup_datetime'].dt.month
fact_trips_cleaned['pickup_year'] = fact_trips_cleaned['tpep_pickup_datetime'].dt.year
fact_trips_cleaned['is_weekend'] = fact_trips_cleaned['pickup_dow'].isin([5, 6]).astype(int)

# Commercial features fields
fact_trips_cleaned['is_airport'] = fact_trips_cleaned['RatecodeID'].isin([2, 3]).astype(int)
fact_trips_cleaned['tip_rate'] = fact_trips_cleaned['tip_amount'] / fact_trips_cleaned['fare_amount']
fact_trips_cleaned['fare_per_mile'] = fact_trips_cleaned['fare_amount'] / fact_trips_cleaned['trip_distance']
fact_trips_cleaned['revenue_per_min'] = fact_trips_cleaned['total_amount'] / fact_trips_cleaned['trip_duration_min']

# Location fields - Segment Fields
fact_trips_cleaned['is_pickup_manhattan'] = fact_trips_cleaned['PU_Borough'].eq('Manhattan')
fact_trips_cleaned['is_dropoff_manhattan'] = fact_trips_cleaned['DO_Borough'].eq('Manhattan')

# Segments
fact_trips_cleaned['segment'] = np.select(
    [   
        fact_trips_cleaned['is_airport'] == 1,
        fact_trips_cleaned['is_pickup_manhattan'] & fact_trips_cleaned['is_dropoff_manhattan'],
        fact_trips_cleaned['is_pickup_manhattan'] ^ fact_trips_cleaned['is_dropoff_manhattan'],
        (~fact_trips_cleaned['is_pickup_manhattan']) & (~fact_trips_cleaned['is_dropoff_manhattan'])
    ],
    [   
        'Airport Related',
        'Manhattan - Manhattan',
        'Manhattan - Non-Manhattan',
        'Non-Manhattan - Non-Manhattan'
    ]
)

# Labels mapping
DOW_LABELS = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
VENDOR_MAP = {1:'Creative Mobile',2:'Curb Mobility',6:'Myle',4:'Vendor4',5:'Vendor5'}
PAYMENT_MAP = {0:'Flex Fare',1:'Credit Card',2:'Cash',3:'No Charge',4:'Dispute',5:'Unknown',6:'Voided'}
RATE_MAP = {1:'Standard',2:'JFK',3:'Newark',4:'Nassau/Westchester',5:'Negotiated',6:'Group',99:'Unknown'}

fact_trips_cleaned['dow_name'] = fact_trips_cleaned['pickup_dow'].map(DOW_LABELS)
fact_trips_cleaned['vendor_name'] = fact_trips_cleaned['VendorID'].map(VENDOR_MAP)
fact_trips_cleaned['payment_name'] = fact_trips_cleaned['payment_type'].map(PAYMENT_MAP)
fact_trips_cleaned['ratecode_name'] = fact_trips_cleaned['RatecodeID'].map(RATE_MAP)

### 3.C Export cleaned and enriched dataset

In [24]:
fact_trips_cleaned.to_csv(r'Datasources\Output\Yellow_Taxi_Assignment_Cleaned_and_Enriched.csv',sep=',',index=False)

## 4. Analysis and Insight Generation


This section brings together the most relevant findings from the dataset and translates them into business-relevant insights.

**Overall, the analysis shows that NYC taxi demand is concentrated in short urban rides, with Manhattan acting as the core mobility hub and weekdays following a clear commuting pattern.**

**At the same time, the data confirms that not all trips behave in the same way: airport rides and Manhattan-related trips form distinct segments with different distance, duration, and pricing dynamics, and should therefore be treated separately in reporting and decision-making.**

**The evolution of demand over time paints a more nuanced picture. Trip volume has declined steadily, including a clear shock during COVID-19, while total amount has continued to rise. This means that revenue growth is being supported more by  pricing effects and longer average distances than by stronger demand, making Trip Count a more reliable indicator of real demand evolution.**

**Beyond demand, the analysis also reveals important commercial dynamics. The vendor landscape is highly concentrated around a dominant player, and tipping behaviour varies meaningfully by payment method and trip segment. Together, these patterns provide a richer view of the market, highlighting differences in competition, customer behaviour, service quality, and potential driver economics.**

### 4.A Trips general profile

The trip profile shows that NYC taxi demand is mainly driven by short urban rides, with most trips taking less than 20 minutes and covering under 5 miles.

Demand also follows a clear temporal pattern. Weekdays exhibit a commuting structure with peaks in the morning and evening, while weekends shift toward later hours and appear more leisure-driven, which has direct implications for driver allocation and pricing strategy.

From a geographical perspective, Manhattan dominates the network as the main origin and destination hub. This confirms that the core of the business is concentrated in the city center.

#### 4.A.I Trip duration and distance distribution

The distribution of trip duration and trip distance shows that the vast majority of NYC taxi rides are short, with most trips lasting less than 20 minutes and covering less than 5 miles. Both variables are strongly right-skewed, which confirms that the core of the business is concentrated in short urban trips rather than long-haul journeys.

This pattern is important from an operational perspective because it suggests that the dataset is dominated by high-frequency, low-distance rides. As a result, average values alone would be misleading, and median or percentile-based measures provide a more accurate view of the typical trip profile.

In [25]:
fact_trips_cleaned[['trip_duration_min', 'trip_distance']].describe().T

,count,mean,std,min,25%,50%,75%,max
trip_duration_min,298824.0,14.247422,11.40942,0.016667,6.70,11.066667,18.016667,121.466667
trip_distance,298824.0,3.151042,3.95239,0.010000,1.04,1.760000,3.250000,38.790000



![alt text](Images\trip_duration_distance_distribution.png?v=2)



#### 4.A.II Hour of Day and Day of the week behaviour

The heatmap reveals a clear difference between weekday and weekend demand patterns. On weekdays, trip volume follows a double-peak structure, with a strong morning commute around 8 AM and a second peak around 6 PM, which is consistent with work-related travel.

On weekends, demand shifts later in the day. Trip activity starts rising in the late morning and reaches its highest levels during the evening and late-night hours, roughly between 10 PM and 2 AM, which suggests that demand is being driven by leisure activities.

From an operational perspective, this means that driver allocation, monitoring intensity, and pricing strategies should not be applied uniformly across the week. Weekdays are more aligned with commuting demand, while weekends require more coverage during late hours and may benefit from a different surge and supply strategy. _A more detailed analysis about this topic is done in the Travel Segment section._

![alt text](Images\hour_day_heatmap.png)

#### 4.A.III Trip distribution across boroughs

The borough-level analysis shows a strong concentration of trips to and from Manhattan, which clearly acts as the city’s main mobility hub. This is consistent with Manhattan being the central business, commercial, and tourist area of NYC, where trip demand is naturally denser than in the outer boroughs.


In [26]:
print("Manhattan is part of Top 4 more busy routes.\n")
borough_profile = sqldf("SELECT PU_Borough, DO_Borough, COUNT(*) AS trip_count FROM fact_trips_cleaned GROUP BY PU_Borough,DO_Borough ORDER BY trip_count DESC", locals())
borough_profile.head()

Manhattan is part of Top 4 more busy routes.



,PU_Borough,DO_Borough,trip_count
0,Manhattan,Manhattan,251772
1,Queens,Manhattan,10541
2,Manhattan,Queens,8607
3,Manhattan,Brooklyn,7324
4,Queens,Queens,5125


![alt text](Images\borough_trip_distribution.png?v=2)


### 4.B Trip segmentation

The segmentation analysis shows that not all trips behave in the same way, and the dataset is clearly driven by a few structurally different travel patterns. Airport trips stand out as a distinct segment, with much longer distances, durations, and higher fares than regular urban rides, which means they should be analysed separately to avoid distorting the overall picture.

Excluding airport trips, Manhattan also emerges as a key differentiator. Trips fully inside Manhattan behave differently from those connecting Manhattan with the outer boroughs or occurring entirely outside the city centre, confirming that location-based segmentation is necessary to understand demand, pricing, and trip dynamics more accurately.

#### 4.B.I Trips involving airports

Airport-related trips show a distinctly different operating pattern compared with non-airport trips. They are substantially longer in both distance and duration, and they also generate a much higher average fare, which reflects a different travel purpose and cost structure. Airport trips are around 4 times longer in median duration and around 4 times higher in average total amount.

From an analytical and operational perspective, airport trips should therefore be treated separately in both reporting and decision-making. Mixing them with regular urban rides would distort the average trip profile and weaken the interpretation of demand, pricing, and service performance.

In [27]:
# Query made using python and not pandasql because pandasql does not have a built-in MEDIAN function
airport_segment = (
    fact_trips_cleaned
    .groupby('is_airport')
    .agg(
        trips=('VendorID', 'count'),
        median_distance=('trip_distance', 'median'),
        median_duration=('trip_duration_min', 'median'),
        avg_total_amount=('total_amount', 'mean')
    )
    .reset_index()
)


airport_segment['Segment'] = airport_segment['is_airport'].map({
    1: 'Airport-related trips',
    0: 'Non-airport trips'
})

airport_segment[["Segment","trips","median_distance","median_duration","avg_total_amount"]]

,Segment,trips,median_distance,median_duration,avg_total_amount
0,Non-airport trips,291241,1.70,10.833333,17.611668
1,Airport-related trips,7583,18.04,43.100000,71.821208


#### 4.B.II Manhattan-centric vs non-Manhattan trips

Manhattan is the most busy part of New York city centre and concentrates a large share of demand. Excluding airport trips, which form a segment on their own, trips can be grouped by whether both ends are in Manhattan, only one end is in Manhattan, or neither end is in Manhattan. This segmentation helps separate dense urban mobility from boundary-crossing and peripheral travel patterns.

From an analytical perspective, this distinction is important because Manhattan trips are likely to follow different distance, duration, and pricing dynamics than trips involving the outer boroughs.

In [28]:
# Query made using python and not pandasql because pandasql does not have a built-in MEDIAN function
manhattan_segment = (
    fact_trips_cleaned
    .groupby('segment')
    .agg(
        trips=('VendorID', 'count'),
        avg_distance=('trip_distance', 'mean'),
        avg_duration=('trip_duration_min', 'mean'),
        avg_total_amount=('total_amount', 'mean')
    )
    .reset_index()
)
manhattan_segment = manhattan_segment[manhattan_segment['segment']!="Airport Related"].sort_values('avg_distance',ascending=False)
manhattan_segment[['avg_distance', 'avg_duration', 'avg_total_amount']] = manhattan_segment[['avg_distance', 'avg_duration', 'avg_total_amount']].round(2)
manhattan_segment

,segment,trips,avg_distance,avg_duration,avg_total_amount
2,Manhattan - Non-Manhattan,24963,8.12,27.73,37.61
3,Non-Manhattan - Non-Manhattan,14696,7.94,22.85,32.34
1,Manhattan - Manhattan,251582,1.94,11.48,14.77


#### 4.B.III Segment Comparison

The segment comparison confirms that airport-related trips are significantly different from the rest of the network. They have the longest median distance, the longest median duration, and by far the highest average total amount, which reinforces the idea that airport trips behave as a distinct travel segment rather than as an extension of regular urban rides.

Among the non-airport segments, trips between Manhattan and non-Manhattan areas show higher distance, duration, and fare than trips within or outside Manhattan. This is consistent with cross-borough travel being more operationally intensive and commercially more valuable than short intra-Manhattan rides.

By contrast, Manhattan-to-Manhattan trips are the shortest and cheapest segment by far. This segment likely represents the high-frequency urban mobility pattern, while the other segments reflect longer and less frequent trips with different pricing and dynamics.

![alt text](Images\segment_metric_comparison.png?v=2)


The heatmaps show that each segment also follows a different behaviour.
- Airport trips are more spread across the day
- Manhattan-to-Manhattan rides are concentrated during weekday business hours
- Manhattan-to-non-Manhattan trips show a clearer late-day and evening pattern
- Trips outside Manhattan tend to be less concentrated and more irregular, with stronger activity in the morning and evening.

![alt text](Images\segment_behaviour.png?v=2)


### 4.C Demand Evolution

Demand has weakened over time in volume terms, with a clear decline in Trip Count and a temporary shock in 2020 caused by COVID-19. Although total amount has increased steadily, this should be interpreted carefully because part of that growth is driven by longer trips and price effects rather than by stronger underlying demand.

Seasonality is relatively limited in the monthly data, although there is a mild summer decline that may reflect weather-related behaviour. Overall, Trip Count is the better indicator of real demand evolution, while revenue should be read together with trip distance and pricing dynamics.

#### 4.C.I Total amount and Trip Count evolution

Trip volume shows a clear downward trend over time, with a temporary drop in 2020 due to COVID-19 and no full recovery to pre-pandemic levels. This suggests that demand, measured in volume terms, has been weakening across the years.

By contrast, total amount has followed a steady upward trend, with a short decline of around 10% during the pandemic and a relatively quick recovery in 2021. The increase in revenue is therefore not necessarily driven by higher demand, but rather by a combination of price effects and longer trip distances. At the same time, the revenue growth of ~30% between 2018-2022 is also driven by the increase in median trip distance, which rose by about 17% over the same period.

For this reasons, Trip Count is the more reliable indicator when analysing demand evolution in real terms, since it is less affected by inflation. 

_Note: 2023 was excluded from the comparison because only January data is available, so it is not directly comparable with full-year observations._

In [29]:
sqldf("SELECT pickup_year,COUNT(total_amount) AS total_trips,SUM(total_amount) AS total_revenue FROM fact_trips_cleaned GROUP BY pickup_year ORDER BY pickup_year", locals())

,pickup_year,total_trips,total_revenue
0,2018,59365,973542.62
1,2019,59136,1131370.23
2,2020,58304,1045181.81
3,2021,58589,1130104.06
4,2022,58547,1261994.07
5,2023,4883,131667.11


![alt text](Images\demand_evolution.png?v=2)


#### 4.C.II Trip amount seasonality

The monthly trip volume does not show a strong seasonal pattern overall. This suggests that demand is relatively stable across the year, with no clear recurring monthly peaks or troughs.

That said, when focusing on the summer period, there is a slight decline across the years between April-September, which is consistent with the idea that weather conditions may influence taxi usage. However, the effect does not appear to be particularly strong in this dataset.

A more detailed analysis would require weather data at a daily level, especially rain information, since taxi demand is likely to behave differently on rainy days than on dry days. Without that granularity, the seasonal effect can only be interpreted as a weak signal rather than a confirmed driver of demand.

_Note: 2020 was excluded from the comparison because of COVID unique effects over demand._

![alt_text](Images\demand_seasonality.png?v=2)

### 4.D Market and Behavioural Insights

This section complements the main demand analysis with two additional perspectives: the market structure from a vendor point of view and the tipping behaviour across segments and payment methods.

#### 4.D.I Market Structure

The market is highly concentrated around a single dominant vendor, Curb Mobility, which accounts for roughly two thirds of total trips and revenue. It is present across all major trip segments, which confirms that its position is not limited to a niche but extends across the full market.

Creative Mobile Technologies is the only meaningful competitor and captures the remaining third of the market, also with coverage across all segments. Curb’s market share has increased consistently over time, which reinforces its dominance and raises a potential concentration risk for the market structure.

Overall, the vendor landscape shows limited competitive balance, with one clear leader and only one substantial challenger. This suggests that any strategic analysis should consider vendor concentration, competitive dependency, and the implications of a progressively less diversified supply base.

![alt_text](Images\market_share_analysis.png)

Myle shows a much more specialised market profile, with a clear focus on the Non-Manhattan segment and exclusively Flex Fare trips. This suggests a more niche positioning, likely concentrated in trip types that are more common outside the city centre and that tend to have a higher total amount.

Its limited market share may be explained by the fact that its trips only start appearing from 2020, which suggests that the company is still establishing its presence in the market. Even so, its entry is positive from a competitive perspective, as a stronger position in this segment could help increase market diversity and reduce dependence on the two dominant vendors.

Myle's trips do not hold any information about `store and forward` system or Passenger Count, so that there must be a problem with their system or communication.

![alt_text](Images\myles_profile.png)

In [30]:
# Flex Fare show the highest Total Amount value
sqldf("SELECT payment_name, ROUND(AVG(total_amount),2) AS avg_total_fare FROM fact_trips_cleaned GROUP BY payment_name ORDER BY 2 DESC", locals())

,payment_name,avg_total_fare
0,Flex Fare,31.72
1,Credit Card,19.67
2,Dispute,17.69
3,No Charge,17.02
4,Cash,15.52


#### 4.D.II Tip analysis

Tip behaviour is an important commercial signal because it affects driver income and can also reflect service quality if drivers are motivated to increase there revenue by tips. One important caveat is that cash tips are not captured in the dataset, so cash trips are excluded from this analysis to avoid understating or biasing the tipping pattern.

The results show higher tips for credit card payments, with an average tip rate of roughly 25%. This might suggest that card-based transactions are more transparent in the dataset.

Airport trips generate the highest tip amounts in absolute terms, which is likely linked to longer ride duration and the added service expectations around luggage handling. However, they do not have the highest tip rate, because that position belongs to Manhattan-to-Manhattan trips.

In [31]:
# Credit Cards have the highest tip amount and tip rate
sqldf("SELECT payment_name,ROUND(AVG(tip_amount),2) AS tip_amount_average,round(AVG(tip_rate) * 100,2) AS tip_rate_average FROM fact_trips_cleaned WHERE payment_name <> 'Cash' GROUP BY payment_name ORDER BY tip_rate_average DESC", locals())

,payment_name,tip_amount_average,tip_rate_average
0,Credit Card,3.02,25.24
1,Flex Fare,2.32,12.55
2,No Charge,0.00,0.00
3,Dispute,0.00,0.00


In [32]:
# Airport segment show the highest tips amount and Manhattan-Manhattan segment the highest tip rate 
sqldf("SELECT segment,ROUND(AVG(tip_amount),2) AS tip_amount_average,ROUND(AVG(tip_rate) * 100,2) AS tip_rate_average FROM fact_trips_cleaned WHERE payment_name <> 'Cash' GROUP BY segment ORDER BY tip_amount_average DESC", locals())

,segment,tip_amount_average,tip_rate_average
0,Airport Related,11.20,20.79
1,Manhattan - Non-Manhattan,5.38,20.10
2,Non-Manhattan - Non-Manhattan,4.16,16.22
3,Manhattan - Manhattan,2.41,25.55
